# Raster-based collision loss for star domains

This notebook implements a **raster-based** alternative to the analytical `_multi_term_star_exclusion`.
Instead of sampling boundary/interior points and checking containment analytically, we:

1. Create a fixed pixel grid covering the scene.
2. **Soft-rasterize** each star domain: assign each pixel a value in [0, 1] indicating
   membership (using a sigmoid of `r_interp - dist`, so the function is smooth and
   differentiable w.r.t. centers and radii).
3. Compute the **overlap area** between pairs of domains as the sum of pixel-wise
   products of their soft masks.

The result is a scalar loss that is zero when the domains are disjoint and grows with
the overlapping area — with well-defined gradients everywhere.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

jax.config.update("jax_enable_x64", False)

## Soft rasterization of a star domain

A star domain is defined by a center `(cx, cy)` and `K` radii `r[k]` at angles
`θ_k = 2πk/K`.  A point `p` is inside the domain when its distance from the center
is less than the linearly-interpolated boundary radius at the angle of `p`.

We make this **soft** with a sigmoid:

$$m(p) = \sigma\!\left(\frac{r_{\text{interp}}(\angle p) - \|p - c\|}{T}\right)$$

where $T$ is a temperature parameter.  Small $T$ → hard boundary; larger $T$ →
blurred boundary with wider gradient support.

In [ ]:
def _interp_radius_at_angles(radii, query_angles, K):
    """Linearly interpolate star radii at arbitrary query angles.

    Args:
        radii: (..., K) star radii at uniformly-spaced angles.
        query_angles: (*Q) query angles in (-π, π].
        K: number of angular samples.

    Returns:
        (*Q) interpolated radii, broadcasting over leading dims of radii.
    """
    delta_theta = 2 * jnp.pi / K
    frac_idx = (query_angles % (2 * jnp.pi)) / delta_theta  # (*Q)
    idx_lo = jnp.floor(frac_idx).astype(jnp.int32) % K
    idx_hi = (idx_lo + 1) % K
    w_hi = frac_idx - jnp.floor(frac_idx)
    r_lo = radii[..., idx_lo]  # (..., *Q)
    r_hi = radii[..., idx_hi]
    return (1.0 - w_hi) * r_lo + w_hi * r_hi


def soft_rasterize_star(
    centers,
    radii,
    grid_xy,
    temperature=0.05,
):
    """Soft-rasterize S star domains onto a pixel grid.

    For each pixel and each domain, computes a soft membership value in [0, 1]
    using a sigmoid of (r_interp - dist) / temperature.

    Args:
        centers: (S, 2) domain centers.
        radii: (S, K) domain radii at K uniformly-spaced angles.
        grid_xy: (H, W, 2) pixel centre coordinates.
        temperature: sigmoid sharpness; smaller → harder boundary.

    Returns:
        masks: (S, H, W) soft membership values in (0, 1).
    """
    S, K = radii.shape
    H, W, _ = grid_xy.shape

    # diff[s, h, w] = grid_xy[h, w] - centers[s]  →  (S, H, W, 2)
    diff = grid_xy[None, :, :, :] - centers[:, None, None, :]  # (S, H, W, 2)

    # Distance from each center to each pixel  →  (S, H, W)
    dist = jnp.sqrt(jnp.sum(diff ** 2, axis=-1) + 1e-12)

    # Angle from each center to each pixel  →  (S, H, W)
    # Double-where trick: safe for autodiff even at the origin
    rho2 = jnp.sum(diff ** 2, axis=-1)
    safe = rho2 > 0
    dx_safe = jnp.where(safe, diff[..., 0], 1.0)
    dy_safe = jnp.where(safe, diff[..., 1], 0.0)
    alpha = jnp.arctan2(dy_safe, dx_safe)  # (S, H, W)

    # Interpolated boundary radius for each (domain, pixel)  →  (S, H, W)
    # We need to call _interp_radius_at_angles for each domain separately
    # (vmap over the S axis).
    def _interp_one(r_s, alpha_s):
        """r_s: (K,), alpha_s: (H, W) → (H, W)"""
        delta_theta = 2 * jnp.pi / K
        frac_idx = (alpha_s % (2 * jnp.pi)) / delta_theta
        idx_lo = jnp.floor(frac_idx).astype(jnp.int32) % K
        idx_hi = (idx_lo + 1) % K
        w_hi = frac_idx - jnp.floor(frac_idx)
        return (1.0 - w_hi) * r_s[idx_lo] + w_hi * r_s[idx_hi]

    r_interp = jax.vmap(_interp_one)(radii, alpha)  # (S, H, W)

    # Soft membership: sigmoid((r_interp - dist) / T)
    masks = jax.nn.sigmoid((r_interp - dist) / temperature)  # (S, H, W)
    return masks

## Pairwise overlap loss

The overlap area between domains $s$ and $t$ is approximated as:

$$\text{overlap}(s, t) = \text{pixel\_area} \cdot \sum_{p} m_s(p) \cdot m_t(p)$$

The collision loss sums the squared overlap over all masked pairs:

In [ ]:
def make_pixel_grid(x_min, x_max, y_min, y_max, resolution):
    """Build a (H, W, 2) grid of pixel-centre coordinates.

    Args:
        x_min, x_max, y_min, y_max: scene bounding box.
        resolution: number of pixels along the longer axis.

    Returns:
        grid_xy: (H, W, 2) float32 array.
        pixel_area: scalar area of one pixel.
    """
    span_x = x_max - x_min
    span_y = y_max - y_min
    if span_x >= span_y:
        W = resolution
        H = max(1, int(round(resolution * span_y / span_x)))
    else:
        H = resolution
        W = max(1, int(round(resolution * span_x / span_y)))

    xs = np.linspace(x_min, x_max, W, endpoint=False) + (x_max - x_min) / (2 * W)
    ys = np.linspace(y_min, y_max, H, endpoint=False) + (y_max - y_min) / (2 * H)
    gx, gy = np.meshgrid(xs, ys)  # (H, W) each
    grid_xy = np.stack([gx, gy], axis=-1).astype(np.float32)  # (H, W, 2)
    pixel_area = float((x_max - x_min) * (y_max - y_min) / (H * W))
    return grid_xy, pixel_area


def raster_collision_loss(optim_vars, input_params):
    """Raster-based pairwise collision loss for star domains.

    For each pair (s, t) where exclusion_mask[s, t] is True, penalises the
    overlap area computed by soft rasterization.

    optim_vars keys:
        "centers": (S, 2)
        "radii":   (S, K)
    input_params keys:
        "grid_xy":        (H, W, 2) pixel-centre coordinates
        "pixel_area":     scalar area of one pixel
        "exclusion_mask": (S, S) bool — True where (s, t) must not overlap
    Optional:
        "temperature":    sigmoid temperature (default 0.05)
    """
    centers = optim_vars["centers"]  # (S, 2)
    radii = optim_vars["radii"]      # (S, K)
    grid_xy = input_params["grid_xy"]            # (H, W, 2)
    pixel_area = input_params["pixel_area"]      # scalar
    mask = input_params["exclusion_mask"]        # (S, S)
    temperature = input_params.get("temperature", 0.05)

    # Soft masks for all domains  →  (S, H, W)
    masks = soft_rasterize_star(centers, radii, grid_xy, temperature)

    # Pairwise overlap: (S, S, H, W) → sum over pixels → (S, S)
    # Use einsum to avoid materializing the full (S, S, H, W) tensor when S is small.
    # overlap[s, t] = pixel_area * sum_p masks[s, p] * masks[t, p]
    HW = grid_xy.shape[0] * grid_xy.shape[1]
    masks_flat = masks.reshape(masks.shape[0], HW)  # (S, HW)
    overlap = pixel_area * (masks_flat @ masks_flat.T)  # (S, S)

    # Apply mask and accumulate
    loss = jnp.sum(jnp.where(mask, overlap ** 2, 0.0))
    return loss

## Sanity check: visualise soft masks and overlap

Two star domains — one circle-like, one more irregular — placed with varying overlap.

In [ ]:
K = 32
angles = np.linspace(0, 2 * np.pi, K, endpoint=False).astype(np.float32)

# Domain A: near-circle of radius 1 centred at (-0.5, 0)
center_a = np.array([-0.5, 0.0], dtype=np.float32)
radii_a = np.ones(K, dtype=np.float32) * 1.0

# Domain B: slightly elliptical, centred at (0.5, 0)
center_b = np.array([0.5, 0.0], dtype=np.float32)
radii_b = (1.0 + 0.3 * np.cos(2 * angles)).astype(np.float32)

centers = np.stack([center_a, center_b])  # (2, 2)
radii = np.stack([radii_a, radii_b])      # (2, K)

grid_xy, pixel_area = make_pixel_grid(-3, 3, -2.5, 2.5, resolution=200)
print(f"Grid: {grid_xy.shape}, pixel_area={pixel_area:.4f}")

masks = soft_rasterize_star(
    jnp.array(centers),
    jnp.array(radii),
    jnp.array(grid_xy),
    temperature=0.04,
)
masks_np = np.array(masks)  # (2, H, W)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(masks_np[0], origin="lower", cmap="Blues", vmin=0, vmax=1)
axes[0].set_title("Domain A mask")
axes[1].imshow(masks_np[1], origin="lower", cmap="Oranges", vmin=0, vmax=1)
axes[1].set_title("Domain B mask")
overlap_img = masks_np[0] * masks_np[1]
axes[2].imshow(overlap_img, origin="lower", cmap="Reds", vmin=0, vmax=1)
axes[2].set_title(f"Overlap (sum={overlap_img.sum() * pixel_area:.3f})")
plt.tight_layout()
plt.show()

## Loss and gradient as a function of centre separation

We sweep the x-offset between the two centres to verify:
- The loss is large when domains overlap heavily.
- The loss falls to ~0 when they are well separated.
- The gradient is smooth throughout.

In [ ]:
exclusion_mask = np.array([[False, True], [True, False]], dtype=bool)

input_params = {
    "grid_xy": jnp.array(grid_xy),
    "pixel_area": float(pixel_area),
    "exclusion_mask": jnp.array(exclusion_mask),
    "temperature": 0.04,
}

loss_jit = jax.jit(raster_collision_loss)
grad_fn = jax.jit(jax.grad(lambda ov, ip: raster_collision_loss(ov, ip)))

offsets = np.linspace(0.0, 4.0, 60)
losses, grads_cx = [], []

for dx in offsets:
    c = jnp.array([[-dx / 2, 0.0], [dx / 2, 0.0]], dtype=jnp.float32)
    ov = {"centers": c, "radii": jnp.array(radii)}
    losses.append(float(loss_jit(ov, input_params)))
    g = grad_fn(ov, input_params)
    # gradient of loss w.r.t. center_b x-component
    grads_cx.append(float(g["centers"][1, 0]))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
ax1.plot(offsets, losses)
ax1.set_ylabel("Collision loss")
ax1.set_title("Raster collision loss vs. centre separation")
ax2.plot(offsets, grads_cx, color="tab:orange")
ax2.set_ylabel("∂loss/∂cx_B")
ax2.set_xlabel("Centre-to-centre distance")
plt.tight_layout()
plt.show()

## Temperature sensitivity

Lower temperature → sharper boundary → steeper loss; higher temperature → smoother gradient but
non-zero loss even for non-overlapping domains ("bleed" from the soft boundary).

In [ ]:
temperatures = [0.01, 0.05, 0.15, 0.40]
offsets_plot = np.linspace(0.0, 4.0, 80)

fig, ax = plt.subplots(figsize=(8, 4))
for T in temperatures:
    ip = {**input_params, "temperature": T}
    ls = []
    for dx in offsets_plot:
        c = jnp.array([[-dx / 2, 0.0], [dx / 2, 0.0]], dtype=jnp.float32)
        ov = {"centers": c, "radii": jnp.array(radii)}
        ls.append(float(loss_jit(ov, ip)))
    ax.plot(offsets_plot, ls, label=f"T={T}")

ax.set_xlabel("Centre-to-centre distance")
ax.set_ylabel("Collision loss")
ax.set_title("Effect of sigmoid temperature")
ax.legend()
plt.tight_layout()
plt.show()

## Wrapping as an `ObjectiveTerm`

The function already matches the `(optim_vars, input_params) → scalar` signature required
by `ObjectiveTerm`, so it can be dropped into any `OptimizationProblemTemplate` directly:

In [ ]:
from vizopt.base import ObjectiveTerm

raster_collision_term = ObjectiveTerm(
    name="raster_collision",
    compute=raster_collision_loss,
    multiplier=5.0,
)
print(raster_collision_term)

## Simple gradient descent demo

Push two initially-overlapping circular star domains apart using only the raster collision loss.

In [ ]:
import optax

K = 48
angles_demo = np.linspace(0, 2 * np.pi, K, endpoint=False).astype(np.float32)

init_centers = jnp.array([[-0.3, 0.0], [0.3, 0.0]], dtype=jnp.float32)
init_radii = jnp.ones((2, K), dtype=jnp.float32) * 1.0

grid_demo, pa_demo = make_pixel_grid(-4, 4, -3, 3, resolution=128)
ip_demo = {
    "grid_xy": jnp.array(grid_demo),
    "pixel_area": float(pa_demo),
    "exclusion_mask": jnp.array([[False, True], [True, False]]),
    "temperature": 0.06,
}

optim = optax.adam(0.05)
ov = {"centers": init_centers, "radii": init_radii}
opt_state = optim.init(ov)

@jax.jit
def step(ov, opt_state):
    loss, grads = jax.value_and_grad(raster_collision_loss)(ov, ip_demo)
    updates, opt_state = optim.update(grads, opt_state)
    ov = optax.apply_updates(ov, updates)
    # Keep radii positive
    ov = {**ov, "radii": jnp.maximum(ov["radii"], 0.05)}
    return ov, opt_state, loss

history = []
for i in range(400):
    ov, opt_state, loss = step(ov, opt_state)
    if i % 50 == 0:
        print(f"iter {i:4d}  loss={float(loss):.4f}")
    history.append(float(loss))

plt.figure(figsize=(7, 3))
plt.plot(history)
plt.xlabel("Iteration")
plt.ylabel("Raster collision loss")
plt.title("Loss during optimisation")
plt.tight_layout()
plt.show()

In [ ]:
# Visualise final configuration
final_masks = soft_rasterize_star(
    ov["centers"], ov["radii"], jnp.array(grid_demo), temperature=0.06
)
final_masks_np = np.array(final_masks)
init_masks = soft_rasterize_star(
    init_centers, init_radii, jnp.array(grid_demo), temperature=0.06
)
init_masks_np = np.array(init_masks)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, masks_np, title in [
    (axes[0], init_masks_np, "Initial (overlapping)"),
    (axes[1], final_masks_np, "After optimisation"),
]:
    rgb = np.zeros((*masks_np.shape[1:], 3))
    rgb[:, :, 0] = masks_np[0]  # domain 0 → red channel
    rgb[:, :, 2] = masks_np[1]  # domain 1 → blue channel
    # overlap → both channels lit → magenta
    ax.imshow(np.clip(rgb, 0, 1), origin="lower")
    ax.set_title(title)

plt.suptitle("Red=domain A, Blue=domain B, Magenta=overlap", y=1.02)
plt.tight_layout()
plt.show()

print(f"Final centres: A={np.array(ov['centers'][0])}, B={np.array(ov['centers'][1])}")
print(f"Separation: {float(jnp.linalg.norm(ov['centers'][0] - ov['centers'][1])):.3f}")

## `optimize_star_domains_raster`

A drop-in replacement for `star_vs_star.optimize_star_domains` that swaps the analytical
`_multi_term_star_exclusion` for the raster-based `raster_collision_loss`.

The **enclosure** constraint (inner set must stay inside outer set) remains analytical
(`_multi_term_star_enclosure`) because it is already well-behaved and does not suffer
from the "gap between sample points" issue that motivated the raster approach for exclusion.

The pixel grid is built once from the initial configuration and held fixed throughout
optimisation (it is part of `input_params`, not `optim_vars`).

In [ ]:
from vizopt.base import OptimConfig, OptimizationProblemTemplate
from vizopt.radially_convex import (
    _multi_term_area,
    _multi_term_min_radius,
    _multi_term_perimeter,
    _multi_term_smoothness,
)
from vizopt.templates.star_vs_star import (
    _build_exclusion_mask,
    _multi_term_star_enclosure,
    _multi_term_target_area,
    _radius_from_target_area,
    _svg_configuration_star_only,
)


def optimize_star_domains_raster(
    S,
    initial_centers,
    k_angles=64,
    target_areas=None,
    initial_radius=1.0,
    weight_target_area=20.0,
    weight_area=1.0,
    weight_perimeter=0.5,
    weight_exclusion=10.0,
    weight_enclosure=20.0,
    weight_smoothness=1.0,
    enclosures=None,
    enclosure_offset=0.1,
    grid_resolution=128,
    grid_margin=0.5,
    temperature=0.05,
    optim_config=None,
    callback=None,
):
    """Optimise pure star domains using a raster-based collision loss for exclusion.

    Identical interface to `star_vs_star.optimize_star_domains`, but replaces
    the analytical star-vs-star exclusion term with `raster_collision_loss`:
    pairwise overlap is measured as the pixel-wise product of soft domain masks,
    which gives well-defined gradients even for complete overlaps.

    The pixel grid is inferred from `initial_centers` and the largest initial
    radius, then held fixed throughout optimisation.

    Args:
        S: number of star domains.
        initial_centers: (S, 2) starting centers.
        k_angles: angular resolution of each boundary polygon.
        target_areas: list of S values (float or None).
        initial_radius: fallback starting radius for sets without a target area.
        weight_target_area: weight for target-area penalty.
        weight_area: weight for area-minimisation regulariser.
        weight_perimeter: weight for perimeter-minimisation regulariser.
        weight_exclusion: weight for raster collision loss.
        weight_enclosure: weight for analytical star-vs-star enclosure.
        weight_smoothness: weight for adjacent-radii smoothness penalty.
        enclosures: list of (inner_idx, outer_idx) pairs.
        enclosure_offset: minimum inset for enclosure constraints.
        grid_resolution: pixels along the longer axis of the bounding box.
        grid_margin: extra margin (in scene units) around the bounding box.
        temperature: sigmoid sharpness for soft rasterization.
        optim_config: OptimConfig; uses defaults when None.
        callback: optional iteration callback.

    Returns:
        (results, history, problem) same as `optimize_star_domains`.
    """
    angles = np.linspace(0, 2 * np.pi, k_angles, endpoint=False).astype(np.float32)
    initial_centers = np.asarray(initial_centers, dtype=np.float32)

    targets_raw = target_areas if target_areas is not None else [None] * S
    target_arr = np.array(
        [t if t is not None else np.nan for t in targets_raw], dtype=np.float32
    )

    initial_radii = np.zeros((S, k_angles), dtype=np.float32)
    for s in range(S):
        r0 = (
            _radius_from_target_area(target_arr[s], k_angles)
            if np.isfinite(target_arr[s])
            else initial_radius
        )
        initial_radii[s] = r0

    # Derive grid bounds from initial config
    max_r = float(initial_radii.max())
    x_min = float(initial_centers[:, 0].min()) - max_r - grid_margin
    x_max = float(initial_centers[:, 0].max()) + max_r + grid_margin
    y_min = float(initial_centers[:, 1].min()) - max_r - grid_margin
    y_max = float(initial_centers[:, 1].max()) + max_r + grid_margin
    grid_xy, pixel_area = make_pixel_grid(x_min, x_max, y_min, y_max, grid_resolution)
    print(f"Pixel grid: {grid_xy.shape[0]}x{grid_xy.shape[1]}, pixel_area={pixel_area:.4f}")

    enclosure_mask = np.zeros((S, S), dtype=bool)
    for inner, outer in enclosures or []:
        enclosure_mask[inner, outer] = True

    exclusion_mask = _build_exclusion_mask(S, enclosures)

    input_parameters = {
        "angles": angles,
        "target_areas": target_arr,
        "enclosure_mask": enclosure_mask,
        "enclosure_offset": float(enclosure_offset),
        # raster-specific
        "grid_xy": grid_xy,
        "pixel_area": float(pixel_area),
        "exclusion_mask": exclusion_mask,
        "temperature": float(temperature),
    }

    def initialize(_, seed):
        return {
            "centers": initial_centers.copy(),
            "radii": initial_radii.copy(),
        }

    terms = [
        ObjectiveTerm("target_area", _multi_term_target_area, weight_target_area),
        ObjectiveTerm("raster_excl", raster_collision_loss, weight_exclusion),
        ObjectiveTerm("star_enclose", _multi_term_star_enclosure, weight_enclosure),
        ObjectiveTerm("min_radius", _multi_term_min_radius, 10.0),
        ObjectiveTerm("smoothness", _multi_term_smoothness, weight_smoothness),
        ObjectiveTerm("area", _multi_term_area, weight_area),
        ObjectiveTerm("perimeter", _multi_term_perimeter, weight_perimeter),
    ]

    problem = OptimizationProblemTemplate(
        terms=terms,
        initialize=initialize,
        svg_configuration=_svg_configuration_star_only,
    ).instantiate(input_parameters)

    optim_vars, history = problem.optimize(optim_config, callback=callback)

    results = [
        {
            "center": np.array(optim_vars["centers"][s]),
            "radii": np.array(optim_vars["radii"][s]),
            "angles": angles,
        }
        for s in range(S)
    ]
    return results, history, problem

## Demo: three domains with enclosure

- **Domain 0** (blue): target area 2, free to move.
- **Domain 1** (orange): target area 3, free to move.
- **Domain 2** (green, outer): no area target — shrinks to fit its contents.

Constraints:
- Domain 0 ⊂ Domain 2
- Domain 1 ⊂ Domain 2
- Domain 0 and Domain 1 must not overlap (raster collision loss).

In [ ]:
from vizopt.animation import SnapshotCallback, snapshots_to_animated_svg
from IPython.display import SVG

K3 = 64
r0_init = _radius_from_target_area(2.0, K3)
r1_init = _radius_from_target_area(3.0, K3)
r_outer = max(r0_init, r1_init) * 2.5

# Place domains 0 and 1 overlapping slightly; domain 2 starts large and centred.
initial_centers_3 = np.array([
    [-r0_init * 0.4,  0.0],   # 0: left inner
    [ r1_init * 0.4,  0.0],   # 1: right inner (overlaps 0 at start)
    [          0.0,   0.0],   # 2: outer container
], dtype=np.float32)

cb3 = SnapshotCallback(every=20)

results3, history3, problem3 = optimize_star_domains_raster(
    S=3,
    initial_centers=initial_centers_3,
    k_angles=K3,
    target_areas=[2.0, 3.0, None],
    initial_radius=r_outer,
    weight_target_area=15.0,
    weight_area=0.5,
    weight_perimeter=0.3,
    weight_exclusion=8.0,
    weight_enclosure=25.0,
    weight_smoothness=0.5,
    enclosures=[(0, 2), (1, 2)],
    enclosure_offset=0.15,
    grid_resolution=96,
    temperature=0.08,
    optim_config=OptimConfig(n_iters=1500, learning_rate=4e-3),
    callback=cb3,
)

In [ ]:
K3 = results3[0]["angles"].shape[0]
dt3 = 2 * np.pi / K3
for s, res in enumerate(results3):
    r = res["radii"]
    area = 0.5 * np.sin(dt3) * np.sum(r * np.roll(r, -1))
    target = [2.0, 3.0, "—"][s]
    print(f"Domain {s}: area = {area:.3f}  (target: {target})")

In [ ]:
COLORS = ["#4472c4", "#ed7d31", "#70ad47"]
LABELS = ["Domain 0 (target=2)", "Domain 1 (target=3)", "Domain 2 (outer)"]

fig, ax = plt.subplots(figsize=(6, 6))
# Draw outer first so inner domains render on top
for s in [2, 0, 1]:
    res = results3[s]
    cx, cy = res["center"]
    r = res["radii"]
    angs = res["angles"]
    bx = np.append(cx + r * np.cos(angs), cx + r[0] * np.cos(angs[0]))
    by = np.append(cy + r * np.sin(angs), cy + r[0] * np.sin(angs[0]))
    ax.fill(bx, by, color=COLORS[s], alpha=0.20)
    ax.plot(bx, by, color=COLORS[s], lw=2, label=LABELS[s])
    ax.plot(cx, cy, "o", color=COLORS[s], ms=5)

ax.set_aspect("equal")
ax.autoscale_view()
ax.margins(0.1)
ax.legend(loc="upper right", fontsize=9)
ax.set_title("Three star domains: raster exclusion + analytical enclosure")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
svg3 = snapshots_to_animated_svg(
    problem3, cb3.snapshots, fps=10, size=450, history=history3, log_scale=True
)
SVG(data=svg3)